# 🛠️ CATIA V5 Qwen 2.5 0.5B 단일 모델 LLM 질문 대답 성능 평가

## 📌 평가 개요 및 목적
본 노트북은 **Qwen 2.5 0.5B (`Qwen/Qwen2.5-0.5B-Instruct`) 단일 로컬 모델**을 대상으로 CATIA V5 대표 질문 3문항(`CATIA_RAG_Evaluation_Dataset.csv`)에 대해 **사전학습 지식(Direct LLM)**과 **RAG 적용(RAG On)** 시의 대답 성능을 검증합니다.

### 📊 검증 모델 정보
- **모델명**: Qwen 2.5 0.5B
- **HuggingFace Repo**: `Qwen/Qwen2.5-0.5B-Instruct` (Qwen 2.5 0.5B 소형 고효율 오픈소스 LLM)
- **구동 방식**: 100% 로컬 PyTorch GPU/CPU 추론 (API 키 0개)

### 🎯 핵심 검증 포인트
1. **눈으로 문장 확인 (Qualitative)**: 질문별 표준 정답(Ground Truth)과 Qwen 2.5 0.5B의 Direct LLM 답변 및 RAG 답변 문장을 1:1 대조 표로 직접 확인합니다.
2. **정량적 점수 비교 (Quantitative)**: Ground Truth 대비 Direct vs RAG 답변의 **BERTScore (Precision, Recall, F1)** 지표를 계산하여 RAG 적용에 따른 정확도 개선율(+ΔF1)을 점수로 확인합니다.

In [6]:
# 1. 환경 설정 및 주요 모듈 임포트
import os
import sys
import gc
import pandas as pd
import torch
from pathlib import Path
from bert_score import score as bert_score_compute
from IPython.display import HTML, display

# 프로젝트 루트 sys.path 추가
PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / ".." / "src").exists():
    PROJECT_ROOT = (PROJECT_ROOT / "..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rag_chain import RAGPipeline
from src.config import DATA_DIR, CHROMA_DB_DIR

print(f"[System] PyTorch CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[System] Active GPU: {torch.cuda.get_device_name(0)}")


[System] PyTorch CUDA Available: True
[System] Active GPU: NVIDIA GeForce RTX 4070 Laptop GPU


In [7]:
# 2. CATIA_RAG_Evaluation_Dataset.csv에서 대표 질문 3개 선별 로딩
csv_file_path = os.path.join(PROJECT_ROOT, "CATIA_RAG_Evaluation_Dataset.csv")

if os.path.exists(csv_file_path):
    df_raw = pd.read_csv(csv_file_path)
    df_raw.rename(columns={
        "ID": "id",
        "카테고리": "category",
        "질문": "question",
        "Qwen 1.5 환각/실패 예상": "expected_failure",
        "RAG 적용 정답 (Ground Truth)": "ground_truth",
        "RAG 문서 근거": "source_doc"
    }, inplace=True)
    
    # 대표 질문 3개 선별 (Q01: Sketcher, Q02: Part Design Rib/Slot, Q10: Specification Tree)
    selected_ids = ["Q01", "Q02", "Q10"]
    df_dataset = df_raw[df_raw["id"].isin(selected_ids)].copy().reset_index(drop=True)
else:
    raise FileNotFoundError(f"Cannot find '{csv_file_path}' dataset file.")

print(f"[Dataset] Selected {len(df_dataset)} Representative CATIA QA Items:")
display(df_dataset[["id", "category", "question", "ground_truth"]])


[Dataset] Selected 3 Representative CATIA QA Items:


,id,category,question,ground_truth
0,Q01,Sketcher Workbench,CATIA V5 Sketcher Workbench의 Operation Tool Ba...,"교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능은 Corner이며, Duplic..."
1,Q02,Part Design (Rib & Slot),CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Pr...,단면 스케치 프로파일과 Center Curve 경로를 따라 3D 솔리드 파이프/리브...
2,Q10,Basic (Tree & Compass Control),CATIA V5 화면의 Specification Tree(스펙 트리)를 화면에서 숨...,Specification Tree를 켜거나 끄는 단축키는 F3 키이며 3D 공간 회...


In [8]:
# 3. Qwen 2.5 0.5B 로컬 모델 단일 답변 생성 (Direct LLM vs RAG LLM)
MODEL_NAME = "Qwen 2.5 0.5B"
REPO_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print(f"==================================================")
print(f"[Experiment] Initializing Local Model: {MODEL_NAME} ({REPO_ID})")
print(f"==================================================")

direct_answers = []
rag_answers = []
source_citations = []

pipeline = RAGPipeline(model_name=REPO_ID, mode="local")

for idx, row in df_dataset.iterrows():
    qid = row["id"]
    question = row["question"]
    print(f" [{idx+1}/{len(df_dataset)}] Processing {qid}...")
    
    # 1) Direct LLM (RAG Off) 순수 사전학습 지식 답변
    ans_direct = pipeline.answer_direct(question)
    direct_answers.append(ans_direct)
    
    # 2) RAG (RAG On) 매뉴얼 문맥 기반 답변
    res_rag = pipeline.answer_rag(question)
    ans_rag = res_rag["answer"]
    sources = res_rag.get("source_pages", [])
    
    rag_answers.append(ans_rag)
    source_citations.append(", ".join(sources))

print(f"\n[Experiment] {MODEL_NAME} Answer Generation Process Completed!")


[Experiment] Initializing Local Model: Qwen 2.5 0.5B (Qwen/Qwen2.5-0.5B-Instruct)
[LLMFactory] Initializing LLM 'Qwen/Qwen2.5-0.5B-Instruct' in mode='local'...
[LLMFactory Local] Downloading/Loading model 'Qwen/Qwen2.5-0.5B-Instruct' on device 'cuda' (PyTorch Local)...


Device set to use cuda:0


[VectorStore] Initializing Local Embeddings 'sentence-transformers/all-MiniLM-L6-v2'...
[VectorStore] Loading existing Chroma database from: C:\KDT_14\[11]Transformer\project\team-03-project\vect\chroma_db_multimodal
 [1/3] Processing Q01...


c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-p

 [2/3] Processing Q02...


c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-p

 [3/3] Processing Q10...


c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:651: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-p


[Experiment] Qwen 2.5 0.5B Answer Generation Process Completed!


## 👁️ 4. 눈으로 직접 보는 생성 문장 비교 셀 (Qualitative Answer Inspection)
3개 질문에 대해 **표준 정답(Ground Truth)**과 **Qwen 2.5 0.5B 모델의 Direct LLM 답변 문장** vs **RAG 적용 답변 문장**을 눈으로 직접 읽고 비교해 볼 수 있는 대조 표입니다.

In [9]:
# 4. Qwen 2.5 0.5B 생성 문장 1:1 Side-by-Side 시각화 표
css_style = """
<style>
    .eval-table {
        width: 100%;
        border-collapse: collapse;
        font-family: 'Segoe UI', Malgun Gothic, sans-serif;
        font-size: 13px;
        margin-top: 10px;
    }
    .eval-table th {
        background-color: #0f172a;
        color: #ffffff;
        text-align: center;
        padding: 10px;
        border: 1px solid #475569;
        font-weight: 600;
    }
    .eval-table td {
        padding: 10px 12px;
        border: 1px solid #cbd5e1;
        vertical-align: top;
        line-height: 1.5;
        word-break: break-word;
    }
    .col-gt { background-color: #f0fdf4; color: #166534; font-weight: 600; }
    .col-direct { background-color: #fef2f2; color: #991b1b; padding: 8px; border-radius: 4px; border: 1px solid #fecdd3; }
    .col-rag { background-color: #eff6ff; color: #1e40af; padding: 8px; border-radius: 4px; border: 1px solid #bfdbfe; font-weight: 500; }
    .badge { display: inline-block; padding: 3px 8px; border-radius: 4px; font-weight: bold; font-size: 11px; margin-bottom: 6px; }
    .badge-direct { background-color: #fda4af; color: #881337; }
    .badge-rag { background-color: #93c5fd; color: #1e3a8a; }
</style>
"""

html_table = css_style + f"""
<div style="overflow-x: auto; border: 1px solid #cbd5e1; border-radius: 6px;">
    <table class="eval-table">
        <thead>
            <tr>
                <th style="width: 60px;">ID</th>
                <th style="width: 220px;">질문 (Question)</th>
                <th style="width: 260px;">표준 정답 (Ground Truth)</th>
                <th style="width: 320px;">{MODEL_NAME} Direct LLM (RAG Off)</th>
                <th style="width: 320px;">{MODEL_NAME} RAG LLM (RAG On)</th>
            </tr>
        </thead>
        <tbody>
"""

for idx, row in df_dataset.iterrows():
    qid = str(row.get("id", ""))
    q = str(row.get("question", ""))
    gt = str(row.get("ground_truth", ""))
    ans_dir = direct_answers[idx]
    ans_rag = rag_answers[idx]
    src = source_citations[idx]
    
    html_table += f"""
            <tr>
                <td style="text-align: center; font-weight: bold;">{qid}</td>
                <td><b>{q}</b></td>
                <td class="col-gt">{gt}</td>
                <td>
                    <span class="badge badge-direct">[RAG Off / Direct LLM]</span><br/>
                    <div class="col-direct">{ans_dir}</div>
                </td>
                <td>
                    <span class="badge badge-rag">[RAG On / 매뉴얼 참조]</span><br/>
                    <div class="col-rag">{ans_rag}</div><br/>
                    <small style="color: #475569;">📍 <b>참조 근거:</b> {src}</small>
                </td>
            </tr>
    """

html_table += """
        </tbody>
    </table>
</div>
"""

display(HTML(html_table))


ID,질문 (Question),표준 정답 (Ground Truth),Qwen 2.5 0.5B Direct LLM (RAG Off),Qwen 2.5 0.5B RAG LLM (RAG On)
Q01,CATIA V5 Sketcher Workbench의 Operation Tool Bar에서 교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능과 선택 요소를 일정 거리만큼 이동 복사하는 기능의 명칭은 무엇인가요?,"교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능은 Corner이며, Duplicate 모드를 정의한 후 선택 요소를 이동 복사하는 기능은 Translate입니다.","[RAG Off / Direct LLM] CATIA V5 Sketcher Workbench에서 교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능과 선택 요소를 일정 거리만큼 이동 복사하는 기능의 명칭은 다음과 같습니다: 1. ""Connect"" (교차) 2. ""Move and Copy"" (선택 요소 이동 복사) 이 두 명칭 모두 CATIA V5 Sketcher Workbench에서 사용할 수 있는 기본적인 교차 및 복사 기능을 나타냅니다. 이 명칭들은 일반적으로 사용자가 교차선을 연결하거나 선택 요소를 복사하기 위해 사용됩니다.",[RAG On / 매뉴얼 참조] CATIA V5 Sketcher Workbench에서 교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능과 선택 요소를 일정 거리만큼 이동 복사하는 기능의 명칭은 다음과 같습니다: 1. **교차 연결**: 이는 두 선분을 연결하는 기능입니다. 2. **선택 요소 이동 복사**: 이는 선택 요소를 이동하고 복사하는 기능입니다. 이两项操作는 CATIA V5 Sketcher Workbench에서 사용할 수 있는 기본적인 작업을 제공합니다. 📍 참조 근거: part design 실습예제.pdf (P.6)
Q02,CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Profile) 단면을 가이드 경로(Center Curve)를 따라 3D 형상으로 생성하는 기능과 그 반대로 홈을 파내는 기능의 명칭은 무엇인가요?,단면 스케치 프로파일과 Center Curve 경로를 따라 3D 솔리드 파이프/리브 형상을 생성하는 기능은 Rib이며 그 반대로 경로를 따라 홈을 파내는 기능은 Slot입니다.,"[RAG Off / Direct LLM] CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Profile) 단면을 가이드 경로(Center Curve)를 따라 3D 형상으로 생성하는 기능은 ""Center Curve"" 또는 ""Center Path""라고 불립니다. 반대의 경우, 홈을 파내는 기능은 ""Home""라고 불립니다.","[RAG On / 매뉴얼 참조] CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Profile) 단면을 가이드 경로(Center Curve)를 따라 3D 형상으로 생성하는 기능은 ""Center Curve""라는 이름의 기능입니다. 또한, 홈을 파내는 기능은 ""Home""라는 이름의 기능입니다. 📍 참조 근거: part design 실습예제.pdf (P.6)"
Q10,CATIA V5 화면의 Specification Tree(스펙 트리)를 화면에서 숨기거나 다시 켜기 위해 사용되는 키보드 단축키와 3D 나침반의 명칭은 무엇인가요?,Specification Tree를 켜거나 끄는 단축키는 F3 키이며 3D 공간 회전 및 이동에 사용되는 나침반 아이콘의 명칭은 Compass 입니다.,"[RAG Off / Direct LLM] CATIA V5 화면에서 Specification Tree(스펙 트리)을 숨기거나 다시 켜는 방법은 다음과 같습니다: 1. **스펙 트리 숨기기**: - 클릭하여 ""스펙 트리"" (specification tree) 라이브러리를 클릭합니다. - 왼쪽 상단에 있는 ""모든 모델"" 버튼을 클릭합니다. - 이 버튼을 클릭하면 모든 모델을 모두 보여줍니다. 여기서 ""모든 모델"" 버튼을 클릭하면 스킨트리만 표시됩니다. 2. **스펙 트리 재설치하기**: - 클릭하여 ""스펙 트리"" 라이브러리를 클릭합니다. - 왼쪽 상단에 있는 ""모든 모델"" 버튼을 클릭합니다. - 이 버튼을 클릭하면 모든 모델을 모두 보여줍니다. 여기서 ""모든 모델"" 버튼을 클릭하면 스킨트리와 함께 재설치됩니다. 3. **3D 나침반 명칭**: -","[RAG On / 매뉴얼 참조] CATIA V5 화면에서 Specification Tree를 숨기거나 다시 켜는 방법은 다음과 같습니다: 1. **SpecTree Hide**: - Ctrl + Shift + H (Windows) - Command + Shift + H (Mac) 2. **SpecTree Unhide**: - Ctrl + Shift + U (Windows) - Command + Shift + U (Mac) 3. **3D View**: - Ctrl + Alt + D (Windows) - Command + Alt + D (Mac) 이러한 키보드 단축키를 사용하면, SpecTree를 화면에서 숨겨서 다시 보여주는 기능을 제공합니다. 3D 나침반의 명칭은 ""3D View""입니다. 📍 참조 근거: part design 실습예제.pdf (P.6)"


## 📊 5. 정량적 점수 비교 셀 (BERTScore Metric Evaluation)
Ground Truth(표준 정답) 문장 대비 Qwen 2.5 0.5B의 **Direct LLM 문장** 및 **RAG 문장** 간 **BERTScore Precision, Recall, F1** 점수를 계산하여 RAG 적용 시 수치적 정확도 상승률(+ΔF1)을 측정합니다.

In [10]:
# 5. BERTScore 정량 평가 및 점수 비교 표 계산
ground_truths = df_dataset["ground_truth"].tolist()

print(f"[Metric] Computing BERTScore (Precision, Recall, F1) for {MODEL_NAME}...\n")

# Direct LLM vs Ground Truth BERTScore
P_dir, R_dir, F1_dir = bert_score_compute(cands=direct_answers, refs=ground_truths, lang="ko", verbose=False)
# RAG LLM vs Ground Truth BERTScore
P_rag, R_rag, F1_rag = bert_score_compute(cands=rag_answers, refs=ground_truths, lang="ko", verbose=False)

f1_list_dir = F1_dir.tolist()
f1_list_rag = F1_rag.tolist()

avg_f1_dir = float(F1_dir.mean())
avg_f1_rag = float(F1_rag.mean())
f1_improvement = avg_f1_rag - avg_f1_dir

# 1) 대표 질문별 세부 점수 데이터프레임
df_detail_scores = pd.DataFrame({
    "ID": df_dataset["id"],
    "Category": df_dataset["category"],
    "Direct LLM F1 Score": [round(f, 4) for f in f1_list_dir],
    "RAG LLM F1 Score": [round(f, 4) for f in f1_list_rag],
    "F1 Improvement (+ΔF1)": [round(r - d, 4) for d, r in zip(f1_list_dir, f1_list_rag)]
})

# 2) 평균 점수 요약 표
df_summary_score = pd.DataFrame([{
    "Model": MODEL_NAME,
    "Direct LLM Avg F1": f"{avg_f1_dir:.4f}",
    "RAG LLM Avg F1": f"{avg_f1_rag:.4f}",
    "Total F1 Improvement (+ΔF1)": f"+{f1_improvement:.4f}"
}])

print("======================================================================")
print(f"          {MODEL_NAME} BERTScore QA Performance Summary              ")
print("======================================================================")
display(df_summary_score)
print("\n🔹 [질문별 세부 BERTScore F1 점수 대조]")
display(df_detail_scores)


[Metric] Computing BERTScore (Precision, Recall, F1) for Qwen 2.5 0.5B...

          Qwen 2.5 0.5B BERTScore QA Performance Summary              


,Model,Direct LLM Avg F1,RAG LLM Avg F1,Total F1 Improvement (+ΔF1)
0,Qwen 2.5 0.5B,0.6995,0.7237,+0.0242



🔹 [질문별 세부 BERTScore F1 점수 대조]


,ID,Category,Direct LLM F1 Score,RAG LLM F1 Score,F1 Improvement (+ΔF1)
0,Q01,Sketcher Workbench,0.7376,0.7331,-0.0046
1,Q02,Part Design (Rib & Slot),0.7621,0.7634,0.0013
2,Q10,Basic (Tree & Compass Control),0.5987,0.6745,0.0758


In [11]:
# 6. 전체 문장 및 점수 비교 데이터셋 CSV 저장
df_export = pd.DataFrame({
    "ID": df_dataset["id"],
    "Category": df_dataset["category"],
    "Question": df_dataset["question"],
    "Ground Truth": df_dataset["ground_truth"],
    f"{MODEL_NAME} Direct Answer": direct_answers,
    f"{MODEL_NAME} Direct F1": [round(f, 4) for f in f1_list_dir],
    f"{MODEL_NAME} RAG Answer": rag_answers,
    f"{MODEL_NAME} RAG F1": [round(f, 4) for f in f1_list_rag],
    "Source Citation": source_citations
})

output_csv_path = os.path.join(PROJECT_ROOT, "data", "CATIA_Qwen0.5B_QA_Evaluation_Results.csv")
os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
df_export.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
print(f"[Saved] Full evaluation results exported to '{output_csv_path}'\n")
display(df_export)


[Saved] Full evaluation results exported to 'C:\KDT_14\[11]Transformer\project\team-03-project\data\CATIA_Qwen0.5B_QA_Evaluation_Results.csv'



,ID,Category,Question,Ground Truth,Qwen 2.5 0.5B Direct Answer,Qwen 2.5 0.5B Direct F1,Qwen 2.5 0.5B RAG Answer,Qwen 2.5 0.5B RAG F1,Source Citation
0,Q01,Sketcher Workbench,CATIA V5 Sketcher Workbench의 Operation Tool Ba...,"교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능은 Corner이며, Duplic...",CATIA V5 Sketcher Workbench에서 교차하는 두 선분을 부드럽게 ...,0.7376,CATIA V5 Sketcher Workbench에서 교차하는 두 선분을 부드럽게 ...,0.7331,part design 실습예제.pdf (P.6)
1,Q02,Part Design (Rib & Slot),CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Pr...,단면 스케치 프로파일과 Center Curve 경로를 따라 3D 솔리드 파이프/리브...,CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Pr...,0.7621,CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Pr...,0.7634,part design 실습예제.pdf (P.6)
2,Q10,Basic (Tree & Compass Control),CATIA V5 화면의 Specification Tree(스펙 트리)를 화면에서 숨...,Specification Tree를 켜거나 끄는 단축키는 F3 키이며 3D 공간 회...,CATIA V5 화면에서 Specification Tree(스펙 트리)을 숨기거나 ...,0.5987,CATIA V5 화면에서 Specification Tree를 숨기거나 다시 켜는 방...,0.6745,part design 실습예제.pdf (P.6)
